# DistilBERT Fake Review Detection — Yelp — **Focal Loss**
**Model:** `distilbert-base-uncased` (fine-tuned for fake review detection)

**Loss function:** Binary focal loss (γ=2.0, α=0.878) instead of standard cross-entropy.

**Training data:** YelpZip + YelpChi + YelpNYC — **full unbalanced dataset (after cross-dataset deduplication, ~13% fake)** from `yelp_split_imbalanced.csv`.
α=0.878 upweights the minority fake class to compensate for the natural ~1:7 imbalance.

**Architecture:** Identical to `train_yelp_distilbert.ipynb` — only the loss function and training data change.

**Tuned for maximum macro F1:**
- **`MAX_LEN = 256` (not 128):** ~44% of Yelp reviews exceed 128 tokens; at 128 nearly half of every review was truncated. 256 cuts truncation to ~17% — the single biggest accuracy lever.
- **Layer-wise LR decay (LLRD)** + **cosine schedule** with warmup — standard recipe for best transformer fine-tuning.
- **Early stopping** on validation macro F1 (patience=3).

*(Matched to the baseline and contrastive DistilBERT notebooks so the three-way ablation is strictly comparable.)*

---
### Instructions
1. Go to **Runtime → Change runtime type → A100 GPU** before running.
2. Run `prepare_yelp_split_imbalanced.ipynb` first to generate `yelp_split_imbalanced.csv` on your Google Drive.
3. Update the path in **cell 3** if needed.
4. Run all cells in order.
5. Download `yelp_fake_distilbert_focal_onnx.zip`, extract it, and place the folder in `data/yelp_fake_distilbert_focal_onnx/`.

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers accelerate optimum[onnxruntime]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THIS PATH if your split file is in a different location ──
SPLIT_CSV = '/content/drive/MyDrive/yelp_split_imbalanced.csv'

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────
# MAX_LEN 256, not 128: ~44% of Yelp reviews exceed 128 tokens (mean ≈153,
# median ≈113). At 128 nearly half of every review was truncated before the
# model saw it. 256 cuts truncation to ~17% — the single biggest accuracy lever.
MAX_LEN       = 256
BATCH_SIZE    = 64
EPOCHS        = 7
LEARNING_RATE = 2e-5   # base (top) LR; lower layers decayed via LLRD
LLRD_DECAY    = 0.9    # layer-wise LR decay — top layers adapt fast, bottom stay near pretrained
WARMUP_RATIO  = 0.1
PATIENCE      = 3      # early stopping on val macro F1
MODEL_NAME    = 'distilbert-base-uncased'
# ── Focal loss hyperparameters ───────────────────────────────────
FOCAL_GAMMA   = 2.0   # focusing parameter: 0 = standard CE, 2 = standard focal
FOCAL_ALPHA   = 0.878 # genuine fraction ≈ 0.878 — upweights minority fake class
# ─────────────────────────────────────────────────────────────────

In [ ]:
# ── Focal loss implementation ────────────────────────────────────
def focal_loss(logits, labels, gamma=2.0, alpha=0.878):
    """
    Multiclass focal loss for binary classification (2 logits).
    gamma : focusing parameter — 0 = standard CE, higher values focus more
            on hard/misclassified examples.
    alpha : weight for the positive (fake) class.
            Set to genuine_fraction (~0.878) to upweight the minority fake class
            in the naturally imbalanced dataset (~12% fake, ~88% genuine).
    """
    probs = torch.softmax(logits, dim=1)                          # [B, 2]
    p_t   = probs.gather(1, labels.unsqueeze(1)).squeeze(1)       # prob of correct class
    focal_weight = (1.0 - p_t) ** gamma                           # down-weight easy examples
    alpha_t = torch.where(
        labels == 1,
        torch.full_like(p_t, alpha),
        torch.full_like(p_t, 1.0 - alpha),
    )
    ce_loss = F.cross_entropy(logits, labels, reduction='none')   # per-sample CE
    return (alpha_t * focal_weight * ce_loss).mean()

In [ ]:
import pandas as pd

df_split = pd.read_csv(SPLIT_CSV)

X_train = df_split[df_split['split'] == 'train']['text'].tolist()
y_train = df_split[df_split['split'] == 'train']['label'].tolist()
X_val   = df_split[df_split['split'] == 'val']['text'].tolist()
y_val   = df_split[df_split['split'] == 'val']['label'].tolist()
X_test  = df_split[df_split['split'] == 'test']['text'].tolist()
y_test  = df_split[df_split['split'] == 'test']['label'].tolist()

print(f'Train: {len(X_train):,}   Val: {len(X_val):,}   Test: {len(X_test):,}')

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

In [ ]:
print('Tokenising training set ...')
train_dataset = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
print('Tokenising validation set ...')
val_dataset   = ReviewDataset(X_val,   y_val,   tokenizer, MAX_LEN)
print('Tokenising test set ...')
test_dataset  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}   Val batches: {len(val_loader)}   Test batches: {len(test_loader)}')

In [ ]:
print(f'Loading model: {MODEL_NAME} ...')
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(DEVICE)
print('Model loaded.')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def build_llrd_optimizer(model, base_lr, decay, weight_decay=0.01):
    """
    Layer-wise learning-rate decay (LLRD) for DistilBertForSequenceClassification.
      - classifier / pre_classifier heads → base_lr
      - top transformer layer (5)         → base_lr * decay
      - ...                                  ...
      - bottom layer (0)                  → base_lr * decay^6
      - embeddings                        → base_lr * decay^7
    Biases and LayerNorm weights get no weight decay (standard).
    """
    n_layers = getattr(model.config, 'n_layers', 6)  # DistilBERT = 6

    def lr_for(name):
        if name.startswith('distilbert.embeddings'):
            return base_lr * (decay ** (n_layers + 1))
        if 'distilbert.transformer.layer.' in name:
            li = int(name.split('distilbert.transformer.layer.')[1].split('.')[0])
            return base_lr * (decay ** (n_layers - li))
        return base_lr  # pre_classifier + classifier

    groups = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        wd = 0.0 if ('bias' in name or 'norm' in name.lower()) else weight_decay
        groups.append({'params': [p], 'lr': lr_for(name), 'weight_decay': wd})
    return AdamW(groups)


optimizer    = build_llrd_optimizer(model, base_lr=LEARNING_RATE, decay=LLRD_DECAY, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Total steps: {total_steps:,}   Warmup: {warmup_steps:,}   LLRD decay: {LLRD_DECAY}  schedule: cosine')

In [ ]:
import os, shutil

for path in [
    'yelp_fake_distilbert_focal_model',
    'yelp_fake_distilbert_focal_onnx',
    'yelp_fake_distilbert_focal_threshold.json',
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f'Removed old folder : {path}')
    elif os.path.isfile(path):
        os.remove(path)
        print(f'Removed old file   : {path}')
print('Clean.')


def get_probs_and_labels(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            outputs = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
            )
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
    return np.array(all_probs), np.array(all_labels)


def find_best_macro_f1_threshold(probs, labels):
    best_thresh, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.90, 0.01):
        f = f1_score(labels, (probs >= t).astype(int), average='macro', zero_division=0)
        if f > best_f1:
            best_f1 = f
            best_thresh = float(t)
    return best_thresh, best_f1


best_f1       = 0
patience_ctr  = 0
train_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = focal_loss(outputs.logits, labels, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)  # focal replaces CE
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if (step + 1) % 200 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{len(train_loader)} | Loss: {total_loss/(step+1):.4f}')

    avg_loss = total_loss / len(train_loader)

    val_probs, val_labels = get_probs_and_labels(model, val_loader)
    val_thresh, val_macro_f1 = find_best_macro_f1_threshold(val_probs, val_labels)
    val_preds = (val_probs >= val_thresh).astype(int)
    val_acc   = accuracy_score(val_labels, val_preds)

    train_history.append({
        'epoch': epoch, 'loss': avg_loss,
        'val_acc': val_acc, 'val_f1': val_macro_f1, 'val_threshold': val_thresh,
    })
    print(f'\nEpoch {epoch}/{EPOCHS} — Loss: {avg_loss:.4f} | Val Accuracy: {val_acc:.1%} | Val Macro F1: {val_macro_f1:.1%} | Threshold: {val_thresh:.2f}')

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        patience_ctr = 0
        model.save_pretrained('yelp_fake_distilbert_focal_model')
        tokenizer.save_pretrained('yelp_fake_distilbert_focal_model')
        print(f'  Best model saved (Val macro F1 = {best_f1:.1%}, threshold = {val_thresh:.2f})')
    else:
        patience_ctr += 1
        print(f'  No improvement ({patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'  Early stopping triggered. Best Val macro F1: {best_f1:.1%}')
            break
    print()

print(f'\nTraining complete. Best Val macro F1: {best_f1:.1%}')

In [ ]:
best_model = DistilBertForSequenceClassification.from_pretrained('yelp_fake_distilbert_focal_model').to(DEVICE)
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = best_model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        )
        all_preds.extend(torch.argmax(outputs.logits, dim=1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

print('=== Test Results — default threshold (0.50) — for reference only ===')
print('(Official result using validation-tuned threshold follows in the next cell)')
print(classification_report(all_labels, all_preds, target_names=['Genuine', 'Fake']))

In [ ]:
import json

best_model_t = DistilBertForSequenceClassification.from_pretrained('yelp_fake_distilbert_focal_model').to(DEVICE)
best_model_t.eval()

all_probs, all_true = [], []
with torch.no_grad():
    for batch in val_loader:
        outputs = best_model_t(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        )
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_true.extend(batch['labels'].numpy())

all_probs = np.array(all_probs)
all_true  = np.array(all_true)

best_thresh, best_thresh_f1 = 0.5, 0.0
for t in np.arange(0.10, 0.90, 0.01):
    f = f1_score(all_true, (all_probs >= t).astype(int), average='macro', zero_division=0)
    if f > best_thresh_f1:
        best_thresh_f1 = f
        best_thresh    = float(t)

print(f'Default threshold 0.50  →  Val macro F1: {f1_score(all_true, (all_probs >= 0.50).astype(int), average="macro"):.1%}')
print(f'Best    threshold {best_thresh:.2f}  →  Val macro F1: {best_thresh_f1:.1%}')

test_probs, test_true = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = best_model_t(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        )
        probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        test_probs.extend(probs)
        test_true.extend(batch['labels'].numpy())

test_probs = np.array(test_probs)
test_true  = np.array(test_true)
print(f'\nTest set — official result (threshold = {best_thresh:.2f}):')
print(classification_report(test_true, (test_probs >= best_thresh).astype(int), target_names=['Genuine', 'Fake']))

with open('yelp_fake_distilbert_focal_threshold.json', 'w') as f:
    json.dump({'threshold': round(float(best_thresh), 2)}, f, indent=2)
print('Threshold saved: yelp_fake_distilbert_focal_threshold.json')

In [ ]:
print('Exporting to ONNX ...')
from optimum.onnxruntime import ORTModelForSequenceClassification

ort_model = ORTModelForSequenceClassification.from_pretrained('yelp_fake_distilbert_focal_model', export=True)
ort_model.save_pretrained('yelp_fake_distilbert_focal_onnx')
tokenizer.save_pretrained('yelp_fake_distilbert_focal_onnx')
print('ONNX model saved to: yelp_fake_distilbert_focal_onnx/')

In [ ]:
from transformers import pipeline

ort_verify = ORTModelForSequenceClassification.from_pretrained(
    'yelp_fake_distilbert_focal_onnx', provider='CPUExecutionProvider'
)
pipe = pipeline('text-classification', model=ort_verify, tokenizer=tokenizer, device=-1)

samples = [
    'Absolutely loved this place! The food was incredible and service was top notch.',
    'Very disappointed. The food was cold and the staff were rude.',
    'Best restaurant in the city! You must try this place, amazing amazing amazing!!!',
]
for text, result in zip(samples, pipe(samples)):
    label = 'FAKE' if result['label'] == 'LABEL_1' else 'GENUINE'
    print(f'{label} ({result["score"]:.1%}) — "{text[:60]}"')

In [ ]:
import matplotlib.pyplot as plt

epochs_range = [h['epoch'] for h in train_history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_range, [h['loss']    for h in train_history], marker='o')
ax1.set_title('Training Loss (Focal)'); ax1.set_xlabel('Epoch')
ax2.plot(epochs_range, [h['val_acc'] for h in train_history], marker='o', label='Accuracy')
ax2.plot(epochs_range, [h['val_f1']  for h in train_history], marker='s', label='F1')
ax2.set_title('Validation Metrics'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
import shutil, os
from google.colab import files

shutil.copy('yelp_fake_distilbert_focal_threshold.json', 'yelp_fake_distilbert_focal_onnx/threshold.json')

shutil.make_archive('yelp_fake_distilbert_focal_onnx', 'zip', 'yelp_fake_distilbert_focal_onnx')
print('Zipped: yelp_fake_distilbert_focal_onnx.zip')

files.download('yelp_fake_distilbert_focal_onnx.zip')

## Next steps
1. Extract `yelp_fake_distilbert_focal_onnx.zip`
2. Place the extracted folder in your `data/` directory:
```
data/yelp_fake_distilbert_focal_onnx/
    model.onnx
    config.json
    tokenizer.json
    vocab.txt
    threshold.json   ← classification threshold tuned on validation set
    ...
```
3. Run `evaluate_yelp_focal.py` to compare focal loss vs baseline results.